<b><h2><center>Wine Quality Case Study</center></h2></b>

## Problem Statement

### Context:

Wine making is both an art and a science. Whether a wine is rated as average or excellent depends on its physicochemical makeup - acidity, sugar, sulphur dioxide, alcohol and so on - and on the sensory impression of expert tasters. Tasting is slow, subjective and expensive, while laboratory measurements are cheap, fast and repeatable. If chemical measurements can predict the quality score, producers can monitor and control quality much earlier in the production process.

### Objective:

You have been hired as a data scientist by a winery that wants to improve the consistency and quality of its wines. The winery has collated lab test results and expert quality scores for a large number of red wine samples. You have been asked to analyze the data and build a model that can predict wine quality from its chemical properties.

- Which physicochemical properties affect wine quality the most? What recommendations can we give the winemakers based on the analysis?
- How can we use different ensemble techniques - Bagging and Boosting - to build a model to predict the quality score of a wine?

### Data Description:

The dataset contains physicochemical tests of red wine samples (Portuguese "Vinho Verde" red wine). Each sample was rated by wine experts on a scale from 0 (very bad) to 10 (excellent); the scores in this dataset range from 3 to 8.

- fixed acidity: most acids involved with wine (tartaric acid, g/dm³) - these do not evaporate readily
- volatile acidity: amount of acetic acid in the wine (g/dm³); at high levels it can lead to an unpleasant vinegar taste
- citric acid: amount of citric acid (g/dm³); found in small quantities, it can add 'freshness' and flavor
- residual sugar: amount of sugar remaining after fermentation stops (g/dm³)
- chlorides: amount of salt in the wine (g/dm³)
- free sulfur dioxide: the free form of SO₂ (mg/dm³); it prevents microbial growth and oxidation of the wine
- total sulfur dioxide: amount of free + bound forms of SO₂ (mg/dm³)
- density: density of the wine (g/cm³), close to that of water depending on the alcohol and sugar content
- pH: describes how acidic or basic the wine is on a scale from 0 (very acidic) to 14 (very basic); most wines are between 3-4
- sulphates: a wine additive (potassium sulphate, g/dm³) that contributes to SO₂ levels and acts as an antimicrobial and antioxidant
- alcohol: percent alcohol content of the wine (% by volume)
- **quality: the target variable** - quality score given by experts (integer values)

## Importing necessary libraries

**Let's start by importing libraries we need.**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn import metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, train_test_split

## Reading the dataset

**Note:** `winequality.csv` uses a semicolon (`;`) as the separator rather than a comma, so we pass `sep=";"` to `read_csv`.

In [ ]:
# Loading dataset (the file is semicolon-separated, so we pass sep=";")
data = pd.read_csv("winequality.csv", sep=";")

## Overview of the dataset

### View the first 5 rows of the dataset

In [ ]:
data.head()

### Check data types and number of non-null values for each column

In [ ]:
data.info()

- We can see that there are a total of 12 columns and 1,599 rows in the dataset.
- All columns are numeric: 11 columns are of float type and the target `quality` is of integer type.
- The number of non-null values of each column is equal to the number of total rows in the dataset i.e. no null values. We can further confirm this using the `isna()` method.

In [ ]:
data.isna().sum()

- There are no missing values in the data.

In [ ]:
# Checking for duplicate rows
data.duplicated().sum()

- There are 240 fully duplicated rows. Two different wines can legitimately have identical lab measurements and the same score (values are rounded), so we **keep** them, as the data provider does. Keep in mind that when duplicates fall in both the train and test sets, test scores can look slightly optimistic.

### Summary of the dataset

In [ ]:
# Summary of all columns
data.describe().T

- Alcohol has a mean of approx 10.4% and a median of 10.2%, with values ranging from 8.4% to 14.9%.
- Residual sugar, chlorides and sulphates have a maximum far above their 75th percentile (e.g. residual sugar: 75th percentile 2.6 vs max 15.5), so these variables have extreme values on the right.
- Total sulfur dioxide ranges from 6 to 289, with a median of 38, so there are some heavily sulphited wines.
- Density and pH are very tightly distributed (density between 0.990 and 1.004, pH mostly between 3.2 and 3.4).
- The target variable `quality` has a mean of approx 5.64 and a median of 6, with a minimum of 3 and a maximum of 8. We will explore this further.

### Number of unique values in each column

In [ ]:
data.nunique()

- There are no ID or date columns in this dataset, so unlike a typical case study there is nothing that needs to be dropped as a pure identifier.
- `quality` has only 6 unique values (3 to 8), so it is an ordinal score. We will treat it as a numeric target and build **regression** models, as it allows us to use R-squared and RMSE to compare models.

### Number of observations in each quality score

In [ ]:
# Number of observations in each quality score
print(data['quality'].value_counts().sort_index())
print('-'*30)
print(data['quality'].value_counts(normalize=True).sort_index().round(3))

- Most of the wines are of average quality: scores 5 and 6 together account for approx 83% of the data (42.6% and 39.9% respectively).
- Only 10 wines have a score of 3 and 18 wines have a score of 8, so extreme qualities are very rare.
- Approx 13.6% of the wines are of high quality (score 7 or above).

In [ ]:
df = data.copy()

## <a name='link2'>Exploratory Data Analysis (EDA) Summary</a>

### **Note**: The EDA section has been covered multiple times in the previous case studies. In this case study, we will mainly focus on the model building aspects. We will only be looking at the key observations from EDA. The detailed EDA can be found in the <a href = #link1>appendix section</a>.

**The below functions need to be defined to carry out the EDA.**

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

### Univariate analysis

In [ ]:
labeled_barplot(data, "quality", perc=True)

- The target variable is concentrated in the middle: scores 5 and 6 make up most of the observations.
- The distribution is approximately symmetric around 5-6 and there are very few observations at both extremes.

In [ ]:
histogram_boxplot(data, "alcohol")

- Alcohol has a right-skewed distribution (skewness approx 0.86) with the mean (approx 10.42) greater than the median (10.2).
- Most wines have an alcohol content of between 9.5% and 11.1%, and there are a few outliers on the higher end.

In [ ]:
histogram_boxplot(data, "volatile acidity")

- Volatile acidity has a slightly right-skewed distribution with a mean and median of approx 0.52-0.53.
- There are a few outliers on the right i.e. wines with a lot of acetic acid.

### Bivariate analysis

**Correlation Check**

In [ ]:
sns.set(rc={'figure.figsize':(16,10)})
sns.heatmap(data.corr(numeric_only=True),
            annot=True,
            linewidths=.5,
            fmt=".2f",
            center=0,
            cbar=False,
            cmap="Spectral")
plt.show()

In [ ]:
# Correlation of every feature with the target
data.corr(numeric_only=True)['quality'].drop('quality').sort_values(ascending=False)

- Alcohol has the highest positive correlation with quality (approx 0.48), followed by sulphates (approx 0.25) and citric acid (approx 0.23).
- Volatile acidity has the strongest negative correlation with quality (approx -0.39), followed by total sulfur dioxide (approx -0.19) and density (approx -0.17).
- Residual sugar, free sulfur dioxide and pH have almost no linear correlation with quality.
- Fixed acidity is strongly correlated with citric acid (approx 0.67), density (approx 0.67) and pH (approx -0.68), and free and total sulfur dioxide are also correlated with each other (approx 0.67). This is expected, as these features measure related chemistry. Tree-based models are not badly affected by this, so we will keep all the features.

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="alcohol", data=data, palette="Paired")
plt.show()

- The median alcohol content increases steadily with quality: it is approx 9.7% for wines with a score of 5 and approx 11.5% and 12.2% for scores 7 and 8 respectively.
- Better wines tend to have a higher alcohol content.

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="volatile acidity", data=data, palette="Paired")
plt.show()

- Volatile acidity decreases as the quality increases, which is in line with its taste effect (vinegar-like smell).
- Low quality wines (3 and 4) have much higher volatile acidity than high quality wines (7 and 8).

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="sulphates", data=data, palette="Paired")
plt.show()

- Higher quality wines have a higher level of sulphates (median rises with the score), and low quality wines have lower sulphate levels. The extreme sulphate values (up to approx 2.0) are found among wines of average quality (scores 4 to 6).

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="citric acid", data=data, palette="Paired")
plt.show()

- Citric acid is higher in better wines: the average is approx 0.17 for scores 3-4 versus approx 0.38-0.39 for scores 7-8.

In [ ]:
# Average value of each feature for every quality score
data.groupby('quality').mean().T.round(3)

- Looking at the average of every feature by quality: as quality rises, alcohol, sulphates and citric acid go up, while volatile acidity, chlorides and density go down.
- Sweetness (residual sugar) is almost the same for every quality level, so it does not distinguish good wines from bad ones.

## Data Preprocessing

### Outlier Detection and Treatment

In [ ]:
# outlier detection using boxplot
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()


plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

- There are quite a few outliers in the data, especially in residual sugar, chlorides, sulphates and total sulfur dioxide.
- However, we will not treat them as they are proper values from the lab tests (a few wines really are very salty or very high in SO₂), and tree-based models are robust to outliers.

**Actions for data pre-processing:**

* There are no missing values and no ID columns, so no imputation or dropping of columns is needed.
* All the features are numeric, so no encoding is required.
* We will not scale the data because decision trees and ensemble methods based on trees are not sensitive to the scale of features.

### Data Preparation for model building

In [ ]:
# Separating features and the target column
X = data.drop('quality', axis=1)
y = data['quality']

In [ ]:
# Splitting the data into train and test sets in 70:30 ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1, shuffle=True)

In [ ]:
X_train.shape, X_test.shape

- We have 1,119 observations in the train set and 480 observations in the test set.

## Bagging - Model Building and Hyperparameter Tuning

- We'll fit different models on the train data and observe their performance.
- We'll try to improve that performance by tuning some hyperparameters available for that algorithm.
- We'll use GridSearchCv for hyperparameter tuning and `r_2 score` to optimize the model.
- R-square - `Coefficient of determination` is used to evaluate the performance of a regression model. It is the amount of the variation in the output dependent attribute which is predictable from the input independent variables.
- Let's start by creating a function to get model scores, so that we don't have to use the same codes repeatedly.

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

In [ ]:
##  Function to calculate r2_score and RMSE on train and test data
def get_model_score(model, flag=True):
    '''
    model : classifier to predict values of X

    '''
    # defining an empty list to store train and test results
    score_list=[]

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    train_r2=metrics.r2_score(y_train,pred_train)
    test_r2=metrics.r2_score(y_test,pred_test)
    train_rmse=np.sqrt(metrics.mean_squared_error(y_train,pred_train))
    test_rmse=np.sqrt(metrics.mean_squared_error(y_test,pred_test))

    #Adding all scores in the list
    score_list.extend((train_r2,test_r2,train_rmse,test_rmse))

    # If the flag is set to True then only the following print statements will be dispayed, the default value is True
    if flag==True:
        print("R-sqaure on training set : ",metrics.r2_score(y_train,pred_train))
        print("R-square on test set : ",metrics.r2_score(y_test,pred_test))
        print("RMSE on training set : ",np.sqrt(metrics.mean_squared_error(y_train,pred_train)))
        print("RMSE on test set : ",np.sqrt(metrics.mean_squared_error(y_test,pred_test)))

    # returning the list with train and test scores
    return score_list

### Decision Tree Model

In [ ]:
dtree = DecisionTreeRegressor(random_state=1)
dtree.fit(X_train, y_train)

In [ ]:
dtree_model_train_perf = model_performance_regression(dtree, X_train, y_train)
print("Training performance \n",dtree_model_train_perf)

In [ ]:
dtree_model_test_perf = model_performance_regression(dtree, X_test, y_test)
print("Testing performance \n",dtree_model_test_perf)

- The Decision tree model with default parameters is completely overfitting the train data (R-squared of 1.0 on train vs approx 0.02 on test).
- Let's see if we can reduce overfitting and improve performance on test data by tuning hyperparameters.

### Hyperparameter Tuning

In [ ]:
# Choose the type of classifier.
dtree_tuned = DecisionTreeRegressor(random_state=1)

# Grid of parameters to choose from
parameters = {'max_depth': list(np.arange(2,20)) + [None],
              'min_samples_leaf': [1, 3, 5, 7, 10],
              'max_leaf_nodes' : [2, 3, 5, 10, 15] + [None],
              'min_impurity_decrease': [0.001, 0.01, 0.1, 0.0]
             }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.r2_score)

# Run the grid search
grid_obj = GridSearchCV(dtree_tuned, parameters, scoring=scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
dtree_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
dtree_tuned.fit(X_train, y_train)

In [ ]:
dtree_tuned_model_train_perf = model_performance_regression(dtree_tuned, X_train, y_train)
print("Training performance \n",dtree_tuned_model_train_perf)

In [ ]:
dtree_tuned_model_test_perf = model_performance_regression(dtree_tuned, X_test, y_test)
print("Testing performance \n",dtree_tuned_model_test_perf)

- The overfitting is greatly reduced after hyperparameter tuning and the test R-squared has increased from approx 0.02 to approx 0.26.
- RMSE is also reduced on test data (approx 0.77 to approx 0.67), so the model generalizes much better than the decision tree with default parameters.
- However, a single tree is still a weak model for this problem.

**Plotting the feature importance of each variable**

In [ ]:
print(pd.DataFrame(dtree_tuned.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = "Imp", ascending = False))

In [ ]:
feature_names = X_train.columns
importances = dtree_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

- alcohol is by far the most important feature for the tuned decision tree, followed by sulphates and volatile acidity.

### Random Forest Model

In [ ]:
rf_estimator = RandomForestRegressor(random_state=1)
rf_estimator.fit(X_train, y_train)

In [ ]:
rf_estimator_model_train_perf = model_performance_regression(rf_estimator, X_train, y_train)
print("Training performance \n",rf_estimator_model_train_perf)

In [ ]:
rf_estimator_model_test_perf = model_performance_regression(rf_estimator, X_test, y_test)
print("Testing performance \n",rf_estimator_model_test_perf)

- Random forest is a big improvement over the decision tree: the test R-squared is approx 0.46 with RMSE of approx 0.57.
- It is, however, overfitting the train data (train R-squared approx 0.93).
- Let's try to reduce this overfitting by hyperparameter tuning.

### Hyperparameter Tuning

In [ ]:
# Choose the type of classifier.
rf_tuned = RandomForestRegressor(random_state=1)

# Grid of parameters to choose from
parameters = {
                'max_depth':[4, 6, 8, 10, None],
                'max_features': ['sqrt','log2',None],
                'n_estimators': [80, 90, 100, 110, 120]
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.r2_score)

# Run the grid search
grid_obj = GridSearchCV(rf_tuned, parameters, scoring=scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
rf_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
rf_tuned.fit(X_train, y_train)

In [ ]:
rf_tuned_model_train_perf = model_performance_regression(rf_tuned, X_train, y_train)
print("Training performance \n",rf_tuned_model_train_perf)

In [ ]:
rf_tuned_model_test_perf = model_performance_regression(rf_tuned, X_test, y_test)
print("Testing performance \n",rf_tuned_model_test_perf)

- There is only a small change in the result after hyperparameter tuning (test R-squared goes from approx 0.459 to approx 0.467), and the model still overfits the train data to some extent.

In [ ]:
print(pd.DataFrame(rf_tuned.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = "Imp", ascending = False))

In [ ]:
feature_names = X_train.columns
importances = rf_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

- alcohol, sulphates and volatile acidity are the three most important features for the tuned random forest model, in line with the correlation analysis. The importance is spread more evenly over the other features than in the single tree.

## Boosting - Model Building and Hyperparameter Tuning

### AdaBoost Regressor

In [ ]:
ab_regressor = AdaBoostRegressor(random_state=1)
ab_regressor.fit(X_train, y_train)

In [ ]:
ab_regressor_model_train_perf = model_performance_regression(ab_regressor, X_train, y_train)
print("Training performance \n",ab_regressor_model_train_perf)

In [ ]:
ab_regressor_model_test_perf = model_performance_regression(ab_regressor, X_test, y_test)
print("Testing performance \n",ab_regressor_model_test_perf)

- AdaBoost is generalizing well (small gap between train and test R-squared) but it gives lower performance than the random forest, with a test R-squared of approx 0.33.

### Hyperparameter Tuning

In [ ]:
# Choose the type of classifier.
ab_tuned = AdaBoostRegressor(random_state=1)

# Grid of parameters to choose from
parameters = {'n_estimators': np.arange(10,100,10),
              'learning_rate': [1, 0.1, 0.5, 0.01],
              }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.r2_score)

# Run the grid search
grid_obj = GridSearchCV(ab_tuned, parameters, scoring=scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
ab_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
ab_tuned.fit(X_train, y_train)

In [ ]:
ab_tuned_model_train_perf = model_performance_regression(ab_tuned, X_train, y_train)
print("Training performance \n",ab_tuned_model_train_perf)

In [ ]:
ab_tuned_model_test_perf = model_performance_regression(ab_tuned, X_test, y_test)
print("Testing performance \n",ab_tuned_model_test_perf)

- There is only a small improvement after tuning (test R-squared of approx 0.36 and RMSE of approx 0.63).

In [ ]:
print(pd.DataFrame(ab_tuned.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = "Imp", ascending = False))

In [ ]:
feature_names = X_train.columns
importances = ab_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

- alcohol is the most important feature for AdaBoost, followed by volatile acidity and sulphates.

### Gradient Boosting Regressor

In [ ]:
gb_estimator = GradientBoostingRegressor(random_state=1)
gb_estimator.fit(X_train, y_train)

In [ ]:
gb_estimator_model_train_perf = model_performance_regression(gb_estimator, X_train, y_train)
print("Training performance \n",gb_estimator_model_train_perf)

In [ ]:
gb_estimator_model_test_perf = model_performance_regression(gb_estimator, X_test, y_test)
print("Testing performance \n",gb_estimator_model_test_perf)

- Gradient boosting with default parameters is generalizing reasonably well, with a test R-squared of approx 0.37 - better than AdaBoost but not as good as random forest.

### Hyperparameter Tuning

In [ ]:
# Choose the type of classifier.
gb_tuned = GradientBoostingRegressor(random_state=1)

# Grid of parameters to choose from
parameters = {'n_estimators': np.arange(50,200,25),
              'subsample':[0.7,0.8,0.9,1],
              'max_features':[0.7,0.8,0.9,1],
              'max_depth':[3,5,7,10]
              }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.r2_score)

# Run the grid search
grid_obj = GridSearchCV(gb_tuned, parameters, scoring=scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
gb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
gb_tuned.fit(X_train, y_train)

In [ ]:
gb_tuned_model_train_perf = model_performance_regression(gb_tuned, X_train, y_train)
print("Training performance \n",gb_tuned_model_train_perf)

In [ ]:
gb_tuned_model_test_perf = model_performance_regression(gb_tuned, X_test, y_test)
print("Testing performance \n",gb_tuned_model_test_perf)

- The model improves after tuning: the test R-squared increases from approx 0.37 to approx 0.43 and RMSE decreases from approx 0.62 to approx 0.59.
- The tuned model is, however, overfitting the train data (train R-squared approx 0.94 vs test approx 0.43).

In [ ]:
print(pd.DataFrame(gb_tuned.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = "Imp", ascending = False))

In [ ]:
feature_names = X_train.columns
importances = gb_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

- alcohol, volatile acidity and sulphates are again the top 3 features, followed by density.

### XGBoost Regressor

In [ ]:
xgb_estimator = XGBRegressor(random_state=1, verbosity=0)
xgb_estimator.fit(X_train, y_train)

In [ ]:
xgb_estimator_model_train_perf = model_performance_regression(xgb_estimator, X_train, y_train)
print("Training performance \n",xgb_estimator_model_train_perf)

In [ ]:
xgb_estimator_model_test_perf = model_performance_regression(xgb_estimator, X_test, y_test)
print("Testing performance \n",xgb_estimator_model_test_perf)

- Compare the test R-squared and RMSE of XGBoost with default parameters with those of the gradient boosting model above.

### Hyperparameter Tuning

In [ ]:
# Choose the type of classifier.
xgb_tuned = XGBRegressor(random_state=1, verbosity=0)

# Grid of parameters to choose from
parameters = {'n_estimators': [75,100,125,150],
              'subsample':[0.7, 0.8, 0.9, 1],
              'gamma':[0, 1, 3, 5],
              'colsample_bytree':[0.7, 0.8, 0.9, 1],
              'colsample_bylevel':[0.7, 0.8, 0.9, 1]
              }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.r2_score)

# Run the grid search
grid_obj = GridSearchCV(xgb_tuned, parameters, scoring=scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
xgb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
xgb_tuned.fit(X_train, y_train)

In [ ]:
xgb_tuned_model_train_perf = model_performance_regression(xgb_tuned, X_train, y_train)
print("Training performance \n",xgb_tuned_model_train_perf)

In [ ]:
xgb_tuned_model_test_perf = model_performance_regression(xgb_tuned, X_test, y_test)
print("Testing performance \n",xgb_tuned_model_test_perf)

In [ ]:
print(pd.DataFrame(xgb_tuned.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = "Imp", ascending = False))

In [ ]:
feature_names = X_train.columns
importances = xgb_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12,12))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

- Check which features XGBoost finds most important. Here too, we expect alcohol, sulphates and volatile acidity to appear at the top, as they did for the other models.

## Comparing all models

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [dtree_model_train_perf.T, dtree_tuned_model_train_perf.T, rf_estimator_model_train_perf.T, rf_tuned_model_train_perf.T,
    ab_regressor_model_train_perf.T, ab_tuned_model_train_perf.T, gb_estimator_model_train_perf.T, gb_tuned_model_train_perf.T,
    xgb_estimator_model_train_perf.T, xgb_tuned_model_train_perf.T],
    axis=1,
)

models_train_comp_df.columns = [
    "Decision Tree",
    "Decision Tree Tuned",
    "Random Forest Estimator",
    "Random Forest Tuned",
    "Adaboost Regressor",
    "Adaboost Tuned",
    "Gradient Boost Estimator",
    "Gradient Boost Tuned",
    "XGB",
    "XGB Tuned"
]

print("Training performance comparison:")
models_train_comp_df

In [ ]:
# Testing performance comparison

models_test_comp_df = pd.concat(
    [dtree_model_test_perf.T, dtree_tuned_model_test_perf.T, rf_estimator_model_test_perf.T, rf_tuned_model_test_perf.T,
    ab_regressor_model_test_perf.T, ab_tuned_model_test_perf.T, gb_estimator_model_test_perf.T, gb_tuned_model_test_perf.T,
    xgb_estimator_model_test_perf.T, xgb_tuned_model_test_perf.T],
    axis=1,
)

models_test_comp_df.columns = [
    "Decision Tree",
    "Decision Tree Tuned",
    "Random Forest Estimator",
    "Random Forest Tuned",
    "Adaboost Regressor",
    "Adaboost Tuned",
    "Gradient Boost Estimator",
    "Gradient Boost Tuned",
    "XGB",
    "XGB Tuned"
]

print("Testing performance comparison:")
models_test_comp_df

In [ ]:
# Best model on the test set (highest R-squared)
best_model_name = models_test_comp_df.loc["R-squared"].idxmax()
print("Best model on test data:", best_model_name)
models_test_comp_df.loc[["RMSE", "R-squared"]].T.sort_values(by="R-squared", ascending=False)

- The ensemble models clearly outperform a single decision tree. The tuned decision tree reaches a test R-squared of approx 0.26, whereas the tuned random forest, gradient boosting and AdaBoost models reach approx 0.47, 0.43 and 0.36 respectively.
- Random forest (bagging) was the best of the scikit-learn models: its tuned version has the highest test R-squared (approx 0.47) and the lowest RMSE (approx 0.57), and it is also the best in terms of MAE and MAPE. Tuned gradient boosting is second.
- The cell above picks the best model automatically from the test R-squared, so check the XGB columns of your run - the final choice is whichever model has the highest test R-squared.
- The R-squared of approx 0.45-0.5 is moderate: wine quality is a subjective score given by human tasters, so it can't be fully explained by lab measurements alone. An RMSE of approx 0.57 means the typical prediction error is about half a quality point, which is useful for flagging likely-weak batches but not for exact scoring.

In [ ]:
# Mapping model names to the fitted models
all_models = {
    "Decision Tree": dtree, "Decision Tree Tuned": dtree_tuned,
    "Random Forest Estimator": rf_estimator, "Random Forest Tuned": rf_tuned,
    "Adaboost Regressor": ab_regressor, "Adaboost Tuned": ab_tuned,
    "Gradient Boost Estimator": gb_estimator, "Gradient Boost Tuned": gb_tuned,
    "XGB": xgb_estimator, "XGB Tuned": xgb_tuned,
}
best_model = all_models[best_model_name]

# Plot observed and predicted values of the test data for the best model
fig, ax = plt.subplots(figsize=(8, 6))
y_pred = best_model.predict(X_test)
ax.scatter(y_test, y_pred, edgecolors=(0, 0, 1))
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=3)
ax.set_xlabel('Observed')
ax.set_ylabel('Predicted')
ax.set_title("Observed vs Predicted - " + best_model_name)
plt.grid()
plt.show()

- Points are spread around the dashed line, but not tightly. Since the true scores are whole numbers (3 to 8) and the model predicts continuous values, the points form vertical stripes.
- The model tends to over-predict the lowest scores and under-predict the highest scores (predictions are pulled towards the average of 5-6), because there are very few examples of extreme-quality wines.

## Conclusions and Business Recommendations

- We can use this predictive model on the lab results of new batches (all of which are known before bottling) to estimate the quality score early and flag batches that are likely to be of poor quality.
- **Alcohol content** is the most important driver of quality in all the models. Better wines tend to have higher alcohol (approx 11.5% or more for scores of 7 and above), so grape ripeness/harvest timing and fermentation control are key levers to focus on.
- **Volatile acidity** is the strongest negative driver. High levels (acetic acid / vinegar taste) are typical in the lowest-rated wines, so the winery should keep strict control over hygiene, oxygen exposure and storage conditions to avoid acetic acid build-up.
- **Sulphates** and **citric acid** are positively associated with quality, so using them in a controlled way can help preserve and improve wines. Total sulfur dioxide, chlorides and density have a small negative association with quality, so excessively high levels should be avoided.
- Residual sugar and pH have very little relationship with quality in this data, so they can be tuned for the wine style/taste profile without a big effect on the rating.
- The model can be used for quality-control screening: batches predicted to be below approx 5 can be sent for a detailed tasting or corrective action before they are bottled, saving expert tasting time.
- Limitations and next steps: the data has very few examples of very poor (3) and excellent (8) wines, and about 240 repeated rows. Collecting more samples of the extreme-quality wines, checking the effect of dropping duplicates, trying stacking, or treating quality as a classification problem (e.g. good: 7+ vs not good) could improve the usefulness of the model.

## <a name='link1'>Appendix: Detailed Exploratory Data Analysis (EDA)</a>

### Univariate analysis

#### Observations on fixed acidity

In [ ]:
histogram_boxplot(df, "fixed acidity")

- Fixed acidity is right-skewed (skewness approx 0.98) with a mean of approx 8.3 and median of 7.9; there are some outliers on the right (up to 15.9).

#### Observations on citric acid

In [ ]:
histogram_boxplot(df, "citric acid")

- Citric acid has a wide spread between 0 and 1 and many wines have exactly 0, i.e. no citric acid; there is just one outlier.

#### Observations on residual sugar

In [ ]:
histogram_boxplot(data, "residual sugar")

- Residual sugar is heavily right-skewed (skewness approx 4.5) with most of the wines between 1.9 and 2.6 g/dm³, and has many outliers (approx 155 values) up to 15.5.

#### Observations on chlorides

In [ ]:
histogram_boxplot(data, "chlorides")

- Chlorides are heavily right-skewed (skewness approx 5.7) with a median of approx 0.079; there are many outliers (approx 112 values), a few of them very large.

#### Observations on free sulfur dioxide

In [ ]:
histogram_boxplot(df, "free sulfur dioxide")

- Free sulfur dioxide is right-skewed with a median of 14 and a maximum of 72.

#### Observations on total sulfur dioxide

In [ ]:
histogram_boxplot(data, "total sulfur dioxide")

- Total sulfur dioxide is right-skewed (skewness approx 1.5) with a median of 38 and a few extreme values up to 289.

#### Observations on density

In [ ]:
histogram_boxplot(df, "density")

- Density is almost perfectly symmetric (skewness approx 0.07), tightly concentrated around 0.9967.

#### Observations on pH

In [ ]:
histogram_boxplot(df, "pH")

- pH is approximately normally distributed with a mean and median of approx 3.31.

#### Observations on sulphates

In [ ]:
histogram_boxplot(data, "sulphates")

- Sulphates are right-skewed (skewness approx 2.4) with a median of 0.62 and a number of outliers on the right (max 2.0).

#### Observations on alcohol

In [ ]:
histogram_boxplot(df, "alcohol")

- Alcohol is right-skewed (skewness approx 0.86) with most of the wines between 9.5% and 11.1% alcohol.

### Bivariate analysis

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="total sulfur dioxide", data=df, palette="Paired")
plt.show()

- Total sulfur dioxide is higher for average wines (score 5) than for the high-quality wines, which have lower values, although there are some outliers.

In [ ]:
sns.set(rc={'figure.figsize':(12,7)})
sns.boxplot(x="quality", y="density", data=df, palette="Paired")
plt.show()

- Density goes slightly down as the quality increases. This is mostly because of the alcohol content: alcohol is lighter than water, so higher alcohol means lower density.

In [ ]:
sns.pairplot(
    df,
    x_vars=["alcohol", "volatile acidity", "sulphates", "citric acid"],
    y_vars=["quality"],
    height=4,
    aspect=1
    );

- Quality tends to increase with alcohol, sulphates and citric acid and decrease with volatile acidity.
- Since the quality is a whole number, the points fall on horizontal lines.

In [ ]:
sns.set(rc={'figure.figsize':(16,10)})
sns.heatmap(df.corr(numeric_only=True),
            annot=True,
            linewidths=.5,
            fmt=".2f",
            center=0,
            cbar=False,
            cmap="Spectral")
plt.show()

- Alcohol (positive) and volatile acidity (negative) have the highest correlation with quality.
- Fixed acidity is correlated with citric acid, density and pH, and free sulfur dioxide with total sulfur dioxide.

### To jump back to the EDA summary section, click <a href = #link2>here</a>.